# Confirmed Exoplanet Census

Starter notebook for the Kaggle dataset in this folder. It loads `exoplanets.csv` with pandas and walks through quality checks and a first piece of analysis.

**Files:** `exoplanets.csv`, `data_dictionary.csv`.

## 1. Load the data

In [ ]:
import pandas as pd

df = pd.read_csv("exoplanets.csv")
df["disc_year"] = pd.to_numeric(df["disc_year"], errors="coerce")
print(df.shape)

## 2. Shape, types and missing values

In [ ]:
import pandas as pd

print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")

missing = df.isna().mean().mul(100).sort_values(ascending=False)
missing[missing > 0].round(1).to_frame("missing_%").head(15)

## 3. Numeric summary

In [ ]:
df.describe(include='number').T.round(2).head(20)

## 4. The radius valley

Plot the radius distribution split by equilibrium-temperature band.

In [ ]:
import numpy as np

ax = (df.dropna(subset=["pl_rade", "teq_class"])
        .query("pl_rade < 20")
        .hist(column="pl_rade", by="teq_class", bins=60, figsize=(14, 8), sharex=True))
for a in np.ravel(ax):
    a.set_xlabel("Planet radius (Earth radii)")
    a.set_ylabel("Planets")

## 5. Mass–radius diagram

Rocky planets follow a steep mass–radius relation; gas giants flatten out.

In [ ]:
sub = df.dropna(subset=["pl_rade", "pl_bmasse"]).query("pl_rade < 20 and pl_bmasse < 5000")
sub.plot.scatter(x="pl_rade", y="pl_bmasse", c="pl_eqt", cmap="inferno", s=12,
                 logy=True, figsize=(9, 6), colorbar=True)

## 6. Discovery timeline by method

In [ ]:
by_year = (df.dropna(subset=["disc_year"])
             .groupby(["disc_year", "discoverymethod"]).size().unstack(fill_value=0))
by_year.plot.area(figsize=(12, 6), cmap="tab20")

## 7. Habitable-zone candidates

In [ ]:
hz = df[df["in_habitable_zone"].astype(str).str.lower() == "true"]
print(f"{len(hz):,} planets in the conservative habitable zone")
hz.groupby(["star_type_group", "planet_type"]).size().sort_values(ascending=False).head(15)

## Next steps

See `KAGGLE.md` for analysis ideas, caveats and provenance.